# Creador de Mundos RPG — Miniaturas de Armaduras, Piezas y Armas

Genera iconos pintados estilo *Lineage 2* en GPU T4 gratuita (`Lykon/dreamshaper-xl-v2-turbo`).
Por defecto arranca en **`MODO = 'muestra'`** (~1 minuto, **16 imágenes**: 3 conjuntos + 10 piezas sueltas + 3 armas) para que puedas comparar en pantalla ambos estilos antes de lanzar el lote completo:
- `'muestra'`: comparativa rápida (16 imágenes, ~1 min)
- `'sets_54'`: las 54 armaduras del catálogo (~3.5 min)
- `'sets_y_armas'`: 54 armaduras + 100 armas y escudos (~9 min)
- `'todo'`: 54 armaduras + 270 piezas sueltas + 100 armas (~25 min; reutiliza las ya generadas en Drive)

In [ ]:
# 1. Verificar GPU y montar Google Drive
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/Creador-de-mundos-Rpg-Colab'
PROMPTS_PATH = f'{WORK_DIR}/assets/armaduras/prompts_armaduras.json'
SCRIPT_PATH = f'{WORK_DIR}/tools/generar-armaduras-ia.py'
OUT_DIR = '/content/armaduras_out'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(PROMPTS_PATH), f'Falta {PROMPTS_PATH}'
assert os.path.exists(SCRIPT_PATH), f'Falta {SCRIPT_PATH}'
print('Drive listo:', WORK_DIR)

In [ ]:
# 2. Instalar dependencias
!pip -q install -U diffusers transformers accelerate safetensors "pillow<13"

In [ ]:
# 3. Elegir MODO y generar (reutiliza lo que ya esté guardado en Drive)
import shutil
MODO = 'muestra'  # Opciones: 'muestra' | 'sets_54' | 'sets_y_armas' | 'todo'

DRIVE_OUT = f'{WORK_DIR}/assets/armaduras'
if os.path.exists(DRIVE_OUT):
    for sub in ('sets', 'piezas', 'armas'):
        src_sub = f'{DRIVE_OUT}/{sub}'
        if os.path.exists(src_sub):
            shutil.copytree(src_sub, f'{OUT_DIR}/{sub}', dirs_exist_ok=True)

!python "{SCRIPT_PATH}" --prompts "{PROMPTS_PATH}" --out "{OUT_DIR}" --modo "{MODO}" --steps 6 --guidance 2.0

In [ ]:
# 4. Copiar a Drive, mostrar comparativa visual lado a lado y descargar ZIP
import base64, glob, json, shutil
from IPython.display import HTML, display
from google.colab import files

DRIVE_OUT = f'{WORK_DIR}/assets/armaduras'
shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)

with open(f'{OUT_DIR}/manifest.json', 'r', encoding='utf-8') as f:
    m = json.load(f)
print('Manifest:', {'sets': len(m['sets']), 'piezas': len(m['piezas']), 'armas': len(m['armas'])})

def bloque_html(titulo, rutas, ancho=120):
    tarjetas = []
    for p in rutas[:12]:
        if not os.path.exists(p): continue
        with open(p, 'rb') as fp:
            b64 = base64.b64encode(fp.read()).decode('ascii')
        nom = os.path.basename(p)
        tarjetas.append(
            f'<div style="text-align:center;background:#0f172a;padding:6px;border-radius:8px;border:1px solid #334155">'
            f'<img src="data:image/webp;base64,{b64}" width="{ancho}" style="border-radius:6px;display:block;margin:0 auto 4px">'
            f'<div style="font-size:11px;color:#e2e8f0;font-family:sans-serif">{nom}</div></div>'
        )
    if not tarjetas: return ''
    return f'<h3 style="color:#facc15;font-family:sans-serif;margin:12px 0 6px">{titulo}</h3><div style="display:flex;flex-wrap:wrap;gap:8px">{chr(10).join(tarjetas)}</div>'

html = (
    bloque_html('Conjuntos de Armadura (sets/)', sorted(glob.glob(f'{OUT_DIR}/sets/*.webp')), 140) +
    bloque_html('Piezas Individuales (piezas/)', sorted(glob.glob(f'{OUT_DIR}/piezas/*.webp')), 96) +
    bloque_html('Armas y Escudos (armas/)', sorted(glob.glob(f'{OUT_DIR}/armas/*.webp')), 96)
)
display(HTML(html))

zip_path = shutil.make_archive('/content/armaduras_valdris', 'zip', OUT_DIR)
print('ZIP listo:', zip_path)
files.download(zip_path)